# 01 - Découverte et compréhension des données BAAC

Avant de chercher les facteurs associés aux blessures, nous devons comprendre comment les
informations sont organisées. Un accident, un véhicule et une personne ne représentent pas
la même chose dans les fichiers. Cette distinction déterminera la manière de réunir les données
et de compter les observations.

Nous commençons ici par les quatre fichiers de 2024. Le but est d'établir leur structure,
de vérifier les liens entre eux et de repérer les points qui demanderont un traitement particulier.
Les conclusions chiffrées correspondent aux CSV examinés le 16 octobre 2026 ; elles seront
à revoir si les fichiers sources sont remplacés.


## Le projet auquel ce notebook contribue

Notre étude porte sur les usagers impliqués dans les accidents corporels enregistrés en
**France métropolitaine, Corse comprise, entre 2019 et 2024**.

> Quelles caractéristiques sont associées au décès ou à l'hospitalisation de ces usagers,
> et dans quelle mesure peut-on estimer la probabilité de cette issue sur une année ultérieure ?

Nous avons retenu **l'usager comme unité d'analyse**. La future table devra donc conserver
une ligne par personne impliquée, même lorsque plusieurs personnes partagent le même véhicule
ou le même accident. Pour le premier modèle, la cible vaudra 1 en cas de décès ou
d'hospitalisation et 0 pour un usager indemne ou légèrement blessé. Une gravité inconnue
sera traitée séparément.

Ce notebook prépare cette table. Le nettoyage commun aux six années et les jointures finales
seront réalisés après l'audit de cohérence du notebook 02.


## Ce que nous voulons établir ici

À la fin de cette découverte, nous devons pouvoir :

- expliquer ce que contient chacun des quatre fichiers et ce que représente une ligne ;
- identifier les clés qui permettent de relier les informations ;
- dire si ces clés sont complètes, uniques au niveau attendu et correctement appariées ;
- interpréter les catégories de gravité et le regroupement retenu pour notre cible ;
- distinguer les constats vérifiés des questions encore ouvertes pour la préparation.

Les tableaux servent ces points précis. Une conclusion suit les vérifications importantes
pour expliquer ce qu'elles changent dans notre manière d'aborder les données.


## Source et périmètre de cette première lecture

Les données proviennent du fichier **BAAC**, administré par l'ONISR. Les extractions annuelles
sont publiées sur [data.gouv.fr](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024).
Nous utilisons également le document *Description des bases de données annuelles des accidents
corporels de la circulation routière, années 2005 à 2024*, daté du 21 octobre 2025.
La copie du PDF se trouve dans `docs/` ; les pages citées dans le notebook renvoient à ce document.

**Les contrôles ci-dessous portent sur les fichiers complets de 2024**, qui comprennent
la métropole et l'outre-mer. Le filtre métropolitain sera appliqué pendant la préparation.
Les effectifs affichés ici décrivent donc le périmètre des fichiers reçus, pas encore celui
de l'analyse finale.

BAAC recense des accidents corporels enregistrés par les forces de l'ordre. Une personne
indemne de la table `usagers` est impliquée dans un de ces accidents. Elle ne représente
pas un trajet sans accident. Notre étude pourra porter sur la gravité parmi les usagers
impliqués ; estimer un risque d'accident par trajet demanderait d'autres données.


## 1. Charger les fichiers de 2024

Les sources sont rangées dans `data/raw/2024`. Nous lisons les identifiants en texte : ils servent
à reconnaître des observations, pas à effectuer des calculs. Cela préserve aussi les codes
géographiques comme `2A` et `2B` et les éventuels zéros initiaux.

Les CSV sont séparés par des points-virgules. Les autres types de colonnes restent pour
l'instant ceux détectés par pandas ; leur préparation sera examinée ensuite.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 90)


In [2]:
# Le notebook peut être lancé depuis notebooks/ ou depuis la racine du projet.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "2024"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Dossier data/raw/2024 introuvable. Vérifier le dossier de travail du notebook."
    )

table_names = ["caracteristiques", "lieux", "vehicules", "usagers"]
missing_files = [name for name in table_names if not (DATA_DIR / f"{name}.csv").is_file()]
if missing_files:
    raise FileNotFoundError(f"Fichiers absents : {missing_files}")


In [3]:
identifier_types = {
    "Num_Acc": "string", "id_vehicule": "string", "id_usager": "string",
    "num_veh": "string", "dep": "string", "com": "string",
}

tables = {}
for name in table_names:
    tables[name] = pd.read_csv(
        DATA_DIR / f"{name}.csv",
        sep=";", encoding="utf-8-sig", dtype=identifier_types, low_memory=False,
    )

caracteristiques = tables["caracteristiques"]
lieux = tables["lieux"]
vehicules = tables["vehicules"]
usagers = tables["usagers"]


Les quatre fichiers sont maintenant chargés séparément. Nous gardons cette séparation
pendant la découverte pour comprendre leur rôle avant de les réunir. Les CSV sources
restent inchangés.

## 2. Un premier regard sur les quatre tables

Nous comparons le nombre de lignes, le nombre de colonnes et le nombre d'accidents distincts.
Ce premier tableau permet de voir immédiatement si toutes les tables décrivent le même
ensemble d'accidents, mais à des niveaux différents. L'égalité des nombres d'accidents
ne suffit pas encore à prouver qu'il s'agit exactement des mêmes identifiants.


In [17]:
overview = pd.DataFrame([
    {"table": name, "lignes": len(df), "colonnes": df.shape[1],
     "accidents_distincts": df["Num_Acc"].nunique()}
    for name, df in tables.items()
])
display(overview)


,table,lignes,colonnes,accidents_distincts
0,caracteristiques,54402,15,54402
1,lieux,70248,18,54402
2,vehicules,92678,11,54402
3,usagers,125187,16,54402


Les fichiers décrivent **54 402 accidents**, mais les volumes de lignes diffèrent :
**92 678 véhicules**, **125 187 usagers** et **70 248** enregistrements
de lieux. Ces différences ne constituent pas à elles seules des erreurs. Un accident
peut impliquer plusieurs véhicules et plusieurs personnes ; le cas des lieux doit être examiné.

Nous regardons ensuite quelques variables pour situer les informations disponibles.


In [18]:
variables_to_inspect = {
    "caracteristiques": ["Num_Acc", "jour", "mois", "an", "lum", "atm", "dep", "lat"],
    "lieux": ["Num_Acc", "catr", "voie", "surf", "vma"],
    "vehicules": ["Num_Acc", "id_vehicule", "num_veh", "catv"],
    "usagers": ["Num_Acc", "id_usager", "id_vehicule", "catu", "an_nais", "grav"],
}

for name, columns in variables_to_inspect.items():
    print(f"\n{name}")
    display(tables[name][columns].head(3))



caracteristiques


,Num_Acc,jour,mois,an,lum,atm,dep,lat
0,202400000001,25,3,2024,2,5,70,"47,56277000"
1,202400000002,20,3,2024,1,7,21,"47,02109000"
2,202400000003,22,3,2024,2,1,15,"44,90238400"



lieux


,Num_Acc,catr,voie,surf,vma
0,202400000001,3,D438,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),9,30
2,202400000002,4,POTERNE (RUE),9,30



vehicules


,Num_Acc,id_vehicule,num_veh,catv
0,202400000001,155 781 758,A01,7
1,202400000001,155 781 759,B01,14
2,202400000002,155 781 757,A01,10



usagers


,Num_Acc,id_usager,id_vehicule,catu,an_nais,grav
0,202400000001,203 988 581,155 781 758,1,2003.0,3
1,202400000001,203 988 582,155 781 759,1,1997.0,1
2,202400000002,203 988 579,155 781 757,3,1927.0,3


Les informations sont complémentaires : `caracteristiques` décrit le contexte de l'accident,
`lieux` la route, `vehicules` les véhicules impliqués et `usagers` les personnes avec leurs blessures.

Les valeurs numériques ne sont pas toutes des mesures. Par exemple, `lum`, `catv` et `grav`
sont des codes de catégories dont le sens vient de la documentation. De même, les coordonnées
peuvent être lues comme du texte à cause de leur séparateur décimal. Le type détecté par pandas
ne suffit donc pas à choisir un traitement statistique.

## 3. Vérifier ce que représente une ligne

### Accidents, véhicules et usagers

Nous examinons les clés candidates de ces trois tables. Une clé adaptée doit être renseignée
et identifier une seule observation au niveau considéré. Pour les véhicules, le couple
`(Num_Acc, id_vehicule)` permet aussi de vérifier le rattachement à l'accident.


In [6]:
keys_to_check = {
    "caracteristiques": ["Num_Acc"],
    "vehicules": ["Num_Acc", "id_vehicule"],
    "usagers": ["id_usager"],
}

key_rows = []
for name, key in keys_to_check.items():
    df = tables[name]
    key_rows.append({
        "table": name, "clé_examinée": " + ".join(key),
        "lignes": len(df),
        "lignes_avec_clé_incomplète": df[key].isna().any(axis=1).sum(),
        "lignes_avec_clé_répétée": df.duplicated(key, keep=False).sum(),
    })
key_report = pd.DataFrame(key_rows)
display(key_report)

print("id_vehicule est unique dans le fichier véhicules :", vehicules["id_vehicule"].is_unique)


,table,clé_examinée,lignes,lignes_avec_clé_incomplète,lignes_avec_clé_répétée
0,caracteristiques,Num_Acc,54402,0,0
1,vehicules,Num_Acc + id_vehicule,92678,0,0
2,usagers,id_usager,125187,0,0


id_vehicule est unique dans le fichier véhicules : True


Sur ces fichiers, les clés examinées sont complètes et ne se répètent pas. Nous pouvons donc
retenir **une ligne par accident** dans `caracteristiques`, **une ligne par véhicule impliqué**
dans `vehicules` et **une ligne par usager** dans `usagers`.

Le numéro d'accident peut se répéter dans les deux dernières tables : il indique que plusieurs
observations appartiennent au même événement. Pour le voir concrètement, prenons un accident
avec deux personnes rattachées au même véhicule.


In [7]:
example_users = usagers.loc[
    usagers["Num_Acc"].eq("202400000050"),
    ["Num_Acc", "id_usager", "num_veh", "catu", "an_nais", "grav"],
].copy()

example_users["catégorie_usager"] = example_users["catu"].map(
    {1: "Conducteur", 2: "Passager", 3: "Piéton"}
)
display(example_users)


,Num_Acc,id_usager,num_veh,catu,an_nais,grav,catégorie_usager
129,202400000050,203 988 441,A01,2,2004.0,4,Passager
130,202400000050,203 988 443,A01,1,2005.0,4,Conducteur


Cet accident comporte un conducteur et un passager du véhicule A01. Le même `Num_Acc` et
le même `num_veh` apparaissent deux fois, mais les `id_usager` sont différents : ce sont bien
deux personnes. Nous devrons conserver deux lignes après la préparation.

La documentation précise aussi qu'un **piéton est rattaché au véhicule qui l'a heurté**
(p. 7 et 10). Pour un piéton, la catégorie du véhicule associé ne décrit donc pas son propre
mode de déplacement. Cette distinction sera importante pour les comparaisons entre usagers.

### Le cas particulier de `lieux`

Le document décrit le lieu principal de l'accident (p. 3). Nous vérifions malgré tout
combien d'enregistrements portent chaque numéro d'accident dans le fichier reçu.


In [8]:
locations_per_accident = lieux.groupby("Num_Acc").size()
location_distribution = (
    locations_per_accident.value_counts().sort_index()
    .rename_axis("enregistrements_par_accident")
    .to_frame("nombre_accidents")
)
display(location_distribution)

print("Accidents avec plusieurs enregistrements :", locations_per_accident.gt(1).sum())
print("Num_Acc manquants dans lieux :", lieux["Num_Acc"].isna().sum())
print("Doublons exacts supplémentaires dans lieux :", lieux.duplicated().sum())


,nombre_accidents
enregistrements_par_accident,
1,38757
2,15463
3,164
4,17
5,1


Accidents avec plusieurs enregistrements : 15645
Num_Acc manquants dans lieux : 0
Doublons exacts supplémentaires dans lieux : 2


**15 645 accidents ont plusieurs enregistrements dans `lieux`**, soit environ
28.8 % des accidents du fichier. Ces lignes ne peuvent pas toutes
être assimilées à des doublons exacts : certains attributs peuvent différer. Nous examinons
un cas pour comprendre ce que cela implique pour les jointures.


In [9]:
EXAMPLE_ACCIDENT = "202400000005"
example_locations = lieux.loc[
    lieux["Num_Acc"].eq(EXAMPLE_ACCIDENT),
    ["Num_Acc", "voie", "catr", "prof", "surf", "vma"],
]
display(example_locations)

example_user_count = usagers["Num_Acc"].eq(EXAMPLE_ACCIDENT).sum()
print("Usagers impliqués :", example_user_count)
print("Enregistrements de lieux :", len(example_locations))
print("Lignes produites par une jointure directe sur Num_Acc :",
      example_user_count * len(example_locations))


,Num_Acc,voie,catr,prof,surf,vma
6,202400000005,BEDOULE (CHEMIN DE LA),1,1,2,50
7,202400000005,BEDOULE (CHEMIN DE LA),1,2,2,70


Usagers impliqués : 4
Enregistrements de lieux : 2
Lignes produites par une jointure directe sur Num_Acc : 8


Dans cet exemple, quatre usagers sont associés à deux enregistrements de lieux. Les deux
enregistrements portent le même nom de voie, mais indiquent des limitations de 50 et 70 km/h
et des profils de route différents. `vma` correspond à la vitesse maximale autorisée,
pas à la vitesse réelle des véhicules.

Une jointure directe créerait huit lignes pour quatre usagers. Il faudra donc définir
une représentation des lieux avant de les ajouter à la table d'analyse. Nous ne choisissons
pas de règle d'agrégation à ce stade : l'examen des six années doit précéder cette décision.


## 4. Vérifier les correspondances entre les tables

Les clés ont le bon niveau d'unicité. Nous vérifions maintenant qu'elles renvoient bien
à des observations présentes dans les autres fichiers.

### Retrouver les mêmes accidents

Nous comparons les ensembles de numéros d'accident dans les deux sens. Cela permet de repérer
à la fois un accident de `caracteristiques` absent d'une autre table et une référence à un accident
qui n'existerait pas dans `caracteristiques`.


In [10]:
reference_accidents = set(caracteristiques["Num_Acc"].dropna())
coverage_rows = []

for name in ["lieux", "vehicules", "usagers"]:
    accident_ids = set(tables[name]["Num_Acc"].dropna())
    coverage_rows.append({
        "table": name,
        "accidents_de_référence_absents_de_la_table": len(reference_accidents - accident_ids),
        "accidents_absents_des_caractéristiques": len(accident_ids - reference_accidents),
        "lignes_sans_Num_Acc": tables[name]["Num_Acc"].isna().sum(),
    })

coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report)


,table,accidents_de_référence_absents_de_la_table,accidents_absents_des_caractéristiques,lignes_sans_Num_Acc
0,lieux,0,0,0
1,vehicules,0,0,0
2,usagers,0,0,0


Les quatre tables couvrent ici exactement les mêmes numéros d'accident et aucune ligne
n'a de `Num_Acc` manquant. Le lien avec les caractéristiques générales de l'accident
est donc complet sur les fichiers examinés.

### Rattacher chaque usager au bon véhicule

Nous utilisons le couple `(Num_Acc, id_vehicule)`. Le contrôle suivant ne construit pas
la table d'analyse : il vérifie uniquement qu'un usager retrouve un seul véhicule dans
le même accident. Nous comparons également `num_veh`, qui désigne le véhicule à l'intérieur
de cet accident.


In [11]:
vehicle_keys = vehicules[["Num_Acc", "id_vehicule", "num_veh"]].rename(
    columns={"num_veh": "num_veh_côté_véhicules"}
)

user_vehicle_check = usagers[["Num_Acc", "id_vehicule", "num_veh"]].merge(
    vehicle_keys,
    on=["Num_Acc", "id_vehicule"],
    how="left", validate="many_to_one", indicator=True,
)

matched = user_vehicle_check["_merge"].eq("both")
comparable = (
    matched & user_vehicle_check["num_veh"].notna()
    & user_vehicle_check["num_veh_côté_véhicules"].notna()
)
different_vehicle_labels = (
    comparable & user_vehicle_check["num_veh"].ne(user_vehicle_check["num_veh_côté_véhicules"])
)

display(pd.Series({
    "usagers_avant": len(usagers),
    "lignes_après_contrôle": len(user_vehicle_check),
    "usagers_sans_id_vehicule": usagers["id_vehicule"].isna().sum(),
    "usagers_sans_véhicule_correspondant": (~matched).sum(),
    "num_veh_non_comparables": (matched & ~comparable).sum(),
    "num_veh_contradictoires": different_vehicle_labels.sum(),
}, name="résultat_du_contrôle"))


usagers_avant                          125187
lignes_après_contrôle                  125187
usagers_sans_id_vehicule                    0
usagers_sans_véhicule_correspondant         0
num_veh_non_comparables                     0
num_veh_contradictoires                     0
Name: résultat_du_contrôle, dtype: int64

Chaque usager retrouve un véhicule dans le bon accident. Les identifiants `num_veh` sont
comparables et concordent, et le contrôle conserve les **125 187 lignes** d'origine.
Ce résultat nous donne une base fiable pour joindre les informations des véhicules aux personnes
lors de la préparation.

Il reste à regarder la relation dans l'autre sens : certains véhicules peuvent exister
sans usager associé.


In [12]:
keys = ["Num_Acc", "id_vehicule"]
user_vehicle_keys = usagers[keys].drop_duplicates()

vehicle_user_check = vehicules[keys].merge(
    user_vehicle_keys,
    on=keys, how="left", validate="one_to_one", indicator=True,
)
vehicles_without_users = vehicle_user_check.loc[
    vehicle_user_check["_merge"].eq("left_only"), keys
]

print("Véhicules sans usager associé :", len(vehicles_without_users))
display(vehicles_without_users.head())


Véhicules sans usager associé : 24


,Num_Acc,id_vehicule
5,202400000004,155 781 755
811,202400000466,155 780 919
4877,202400002915,155 776 710
8563,202400005069,155 772 924
11886,202400007018,155 769 475


Nous trouvons **24 véhicules sans usager associé**. Le PDF explique précisément cet écart
pour 2024 : des conducteurs en fuite n'ont pas été créés dans la table `usagers` (p. 3).
L'absence d'appariement est donc documentée. Elle ne justifie pas d'inventer une personne
ou une gravité pour compléter les lignes manquantes.

Le contrôle des liens a ainsi deux conclusions différentes : tous les usagers présents
retrouvent leur véhicule, mais tous les véhicules n'ont pas nécessairement d'usager enregistré.


## 5. Comprendre la gravité et repérer les informations inconnues

### Lire correctement `grav`

La documentation distingue quatre situations (p. 2 et 11) :

| Code | Signification |
|---|---|
| 1 | Indemne |
| 2 | Tué : décès sur le coup ou dans les 30 jours suivant l'accident |
| 3 | Blessé hospitalisé : hospitalisation de plus de 24 heures |
| 4 | Blessé léger |

Nous regardons les effectifs et les proportions pour situer la variable. Les nombres sont
des étiquettes : `4` n'est pas plus grave que `2`, et la moyenne de ces codes ne serait pas
un indicateur de gravité interprétable.


In [13]:
severity_labels = {1: "Indemne", 2: "Tué", 3: "Blessé hospitalisé", 4: "Blessé léger"}
severity_counts = usagers["grav"].value_counts(dropna=False).sort_index()
severity_report = severity_counts.rename_axis("code").reset_index(name="effectif")
severity_report["catégorie"] = severity_report["code"].map(severity_labels).fillna("Inconnue / non documentée")
severity_report["pourcentage_des_usagers"] = (
    100 * severity_report["effectif"] / len(usagers)
).round(2)

display(severity_report[["code", "catégorie", "effectif", "pourcentage_des_usagers"]])


,code,catégorie,effectif,pourcentage_des_usagers
0,1,Indemne,52920,42.27
1,2,Tué,3432,2.74
2,3,Blessé hospitalisé,19126,15.28
3,4,Blessé léger,49709,39.71


Les indemnes et les blessés légers représentent la majorité des personnes enregistrées.
Les décès constituent une catégorie beaucoup moins fréquente : un modèle évalué seulement
sur sa proportion globale de bonnes réponses pourrait masquer une mauvaise détection de ces cas.

Nous avons retenu pour le premier modèle le regroupement **« décès ou hospitalisation »**.
Nous en montrons ici la traduction sur 2024, tout en conservant les quatre catégories originales
pour les descriptions. Il s'agit d'un regroupement de travail, pas d'une définition médicale
des blessures graves.


In [14]:
# Aperçu du regroupement, sans modifier la colonne grav ni créer la table finale.
target_preview = usagers["grav"].map({1: 0, 2: 1, 3: 1, 4: 0}).astype("Int64")

target_report = pd.DataFrame({
    "groupe": ["0 - Indemne ou blessé léger", "1 - Tué ou hospitalisé", "Gravité inconnue / non documentée"],
    "effectif": [target_preview.eq(0).sum(), target_preview.eq(1).sum(), target_preview.isna().sum()],
})
target_report["pourcentage_des_usagers"] = (100 * target_report["effectif"] / len(usagers)).round(2)
display(target_report)


,groupe,effectif,pourcentage_des_usagers
0,0 - Indemne ou blessé léger,102629,81.98
1,1 - Tué ou hospitalisé,22558,18.02
2,Gravité inconnue / non documentée,0,0.00


Sur le périmètre complet des fichiers 2024, ce regroupement donne **22 558 cas positifs**
et **102 629 cas négatifs**. La proportion de cas positifs est d'environ
18.0 %. Les quatre modalités de `grav` sont renseignées pour tous les
usagers de ce fichier ; ce constat devra être vérifié sur les autres années. Ces proportions
seront recalculées après le filtre métropolitain.

La documentation signale une modification de saisie de la catégorie « hospitalisé » depuis
2018 et l'absence de labellisation de cet indicateur depuis 2019 (p. 1). Nous garderons cette
limite dans l'interprétation et examinerons aussi les décès séparément. Une gravité inconnue
ne sera jamais assimilée à une issue négative par défaut.

### Les valeurs manquantes ne se limitent pas aux NA

Nous commençons par les cellules que pandas reconnaît comme manquantes à la lecture.
Le dénominateur est, pour chaque colonne, le nombre de lignes de sa table.


In [15]:
missing_rows = []
for name, df in tables.items():
    for column in df.columns:
        count = df[column].isna().sum()
        if count > 0:
            missing_rows.append({
                "table": name, "variable": column,
                "NA_à_la_lecture": count,
                "pourcentage": round(100 * count / len(df), 2),
            })

missing_report = pd.DataFrame(missing_rows)
display(missing_report.sort_values(["table", "pourcentage"], ascending=[True, False]))


,table,variable,NA_à_la_lecture,pourcentage
0,caracteristiques,adr,2310,4.25
3,lieux,lartpc,70215,99.95
2,lieux,v2,64332,91.58
1,lieux,voie,13331,18.98
5,usagers,an_nais,2579,2.06
4,vehicules,occutc,91729,98.98


Ce tableau ne mesure pas à lui seul toute l'information inconnue. Certaines colonnes utilisent
des codes particuliers. Pour le vérifier, nous comptons quelques valeurs effectivement présentes,
sans les recoder à ce stade.


In [16]:
codes_to_inspect = [
    ("caracteristiques", "col", -1),
    ("usagers", "sexe", -1),
    ("usagers", "trajet", 0),
    ("usagers", "secu1", 0),
]

special_code_rows = []
for name, column, value in codes_to_inspect:
    values = tables[name][column]
    special_code_rows.append({
        "table": name, "variable": column, "code_observé": value,
        "effectif_du_code": values.eq(value).sum(),
        "NA_à_la_lecture": values.isna().sum(),
    })
display(pd.DataFrame(special_code_rows))


,table,variable,code_observé,effectif_du_code,NA_à_la_lecture
0,caracteristiques,col,-1,6,0
1,usagers,sexe,-1,2395,0
2,usagers,trajet,0,35024,0
3,usagers,secu1,0,12216,0


Le sens du code dépend de la variable. Le PDF définit `trajet = 0` comme « non renseigné »,
alors que `secu1 = 0` signifie « aucun équipement ». Pour `sexe`, les valeurs décrites sont
1 et 2 ; le code -1 observé doit donc être signalé comme non décrit dans cette rubrique.
Nous ne pouvons pas transformer tous les zéros, ni tous les codes particuliers, avec une seule règle.

Le notebook 02 étendra ce rapprochement entre documentation et valeurs observées aux six années.
Les règles de préparation seront définies ensuite, variable par variable.

## 6. Ce que cette découverte nous permet de conclure

Nous avons maintenant identifié les niveaux d'information et vérifié les correspondances
sur les fichiers 2024. Le tableau suivant relie ces constats au travail de préparation.

| Constat sur 2024 | Conséquence pour le projet |
|---|---|
| Les clés examinées sont complètes et uniques aux niveaux accident, véhicule et usager | Ces clés peuvent servir de base aux contrôles des jointures |
| Les quatre tables couvrent le même ensemble d'accidents | Aucun écart de couverture des accidents n'est observé sur ces fichiers |
| Chaque usager retrouve son véhicule, sans multiplication des lignes | Les informations du véhicule pourront être ajoutées avec un contrôle de cardinalité |
| 24 véhicules n'ont pas d'usager associé ; cet écart est expliqué par le PDF | Conserver cette particularité dans la documentation, sans inventer d'observation |
| Certains accidents ont plusieurs enregistrements de lieux avec des attributs différents | Définir une représentation des lieux avant la jointure finale |
| `grav` comporte quatre catégories dont les codes ne sont pas une échelle | Conserver les libellés et appliquer explicitement le regroupement choisi |
| Des codes spéciaux portent des sens différents selon les colonnes | Distinguer information inconnue, sans objet et absence réelle d'un équipement ou d'un aménagement |

Ces vérifications portent sur la structure et les correspondances. Elles ne démontrent
ni l'absence d'erreurs dans toutes les valeurs, ni une stabilité de la collecte entre années.
Nous ne tirons pas encore de conclusion sur les facteurs associés aux blessures.

### Les choix déjà faits et les questions encore ouvertes

Le périmètre **métropolitain**, l'unité **usager** et la cible **décès ou hospitalisation** sont fixés
dans le cahier des charges. La table finale devra appliquer ces choix tout en gardant une trace
des observations sources et des exclusions.

Il reste à examiner la cohérence sur 2019-2024, à définir le traitement des lieux multiples
et à arrêter les règles des valeurs inconnues ou atypiques. Ces décisions doivent précéder
la construction de la table finale.

### Passage au notebook 02

La suite consiste à vérifier si les schémas, les clés et les conventions observés ici restent
comparables sur les six années. Le notebook 02 fournira une liste de transformations justifiées
pour le notebook 03, consacré à la préparation et aux jointures.

Pour la comparaison prédictive, l'apprentissage portera sur 2019-2022, la validation sur 2023
et le test sur 2024. La découverte et l'audit déjà réalisés de 2024 seront déclarés ; les scores
de cette année ne serviront pas à régler le modèle. Les usagers d'un même accident sont liés,
ce qui devra aussi être pris en compte dans les partitions et les méthodes d'incertitude concernées.

À ce stade, nous devons pouvoir expliquer ce que représente une ligne dans chaque fichier,
comment une personne retrouve son accident et son véhicule, pourquoi les lieux demandent
une précaution particulière et comment la gravité sera définie pour la suite.
